In [1]:
import pandas as pd

def load_electricitymaps(filepath, region_name):
    df = pd.read_csv(filepath)
    df["datetime"] = pd.to_datetime(df["Datetime (UTC)"])
    df = df[["datetime",
             "Carbon intensity gCO₂eq/kWh (direct)",
             "Carbon-free energy percentage (CFE%)",
             "Renewable energy percentage (RE%)"]]
    df.columns = ["datetime", "carbon_intensity", "cfe_pct", "re_pct"]
    df["region"] = region_name
    return df

def load_openmeteo(filepath):
    df = pd.read_csv(filepath, skiprows=3)
    df["datetime"] = pd.to_datetime(df["time"])
    df = df[["datetime",
             "temperature_2m (°C)",
             "wind_speed_10m (km/h)",
             "cloud_cover (%)",
             "shortwave_radiation (W/m²)"]]
    df.columns = ["datetime", "temp_c", "wind_speed", "cloud_cover", "solar_radiation"]
    return df

# load ElectricityMaps
em_california = load_electricitymaps("snapshots_2026-02-10_US-CAL-CISO-2023-hourly.csv", "california")
em_texas      = load_electricitymaps("snapshots_2026-02-10_US-TEX-ERCO-2023-hourly.csv", "texas")
em_illinois   = load_electricitymaps("snapshots_2026-02-10_US-MIDW-MISO-2023-hourly.csv", "illinois")
em_virginia   = load_electricitymaps("snapshots_2026-02-10_US-MIDA-PJM-2023-hourly.csv", "virginia")

# load Open-Meteo
wx_california = load_openmeteo("open-meteo-california.csv")
wx_texas      = load_openmeteo("open-meteo-texas.csv")
wx_illinois   = load_openmeteo("open-meteo-illinois.csv")
wx_virginia   = load_openmeteo("open-meteo-virginia.csv")

# merge each region
def merge_region(em, wx):
    return pd.merge(em, wx, on="datetime", how="inner")

california = merge_region(em_california, wx_california)
texas      = merge_region(em_texas,      wx_texas)
illinois   = merge_region(em_illinois,   wx_illinois)
virginia   = merge_region(em_virginia,   wx_virginia)

# stack all regions
df = pd.concat([california, texas, illinois, virginia], ignore_index=True)

# add time features
df["hour"]        = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.dayofweek
df["month"]       = df["datetime"].dt.month


print("Shape:", df.shape)
print("Rows per region:\n", df["region"].value_counts())
print("Null counts:\n", df.isnull().sum())

# save
df.to_csv("merged_dataset.csv", index=False)
print("Saved to merged_dataset.csv")

Shape: (35040, 12)
Rows per region:
 region
california    8760
texas         8760
illinois      8760
virginia      8760
Name: count, dtype: int64
Null counts:
 datetime            0
carbon_intensity    0
cfe_pct             0
re_pct              0
region              0
temp_c              0
wind_speed          0
cloud_cover         0
solar_radiation     0
hour                0
day_of_week         0
month               0
dtype: int64
Saved to merged_dataset.csv
